In [3]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

#Gerador JSON - Wolf Troubleshooting
#Versão v5 - suporta o novo modelo de diagnóstico guiado inspirado no padrão Cummins.

#Estrutura padrão local:
#C:\Users\wesley\Documents\13 - TroubleShooting\
#├── 00 - Python\gerar_json_troubleshooting.py
#├── 01 - DB\Base_Pai_Troubleshooting_Alertas_Falhas_MVP.xlsx
#└── 99 - Saidas JSON\dados_troubleshooting.json


from __future__ import annotations

import argparse
import hashlib
import json
import re
from datetime import datetime, date
from pathlib import Path
from typing import Any

import openpyxl

ROOT = Path(r"C:\Users\wesley\Documents\13 - TroubleShooting")
DEFAULT_EXCEL = ROOT / r"01 - DB\Base_Pai_Troubleshooting_Alertas_Falhas_MVP.xlsx"
DEFAULT_SAIDA = ROOT / r"99 - Saidas JSON"

SHEET_TROUBLE = "01_Troubleshooting"
SHEET_DIAG = "02_Diagnostico_Guiado"
SHEET_ANEXOS = "03_Anexos"
SHEET_REVISOES = "04_Revisoes"
SHEET_LISTAS = "Listas"


def slug_header(value: Any) -> str:
    text = str(value or "").strip().lower()
    repl = {
        "ç": "c", "ã": "a", "á": "a", "à": "a", "â": "a", "ä": "a",
        "é": "e", "è": "e", "ê": "e", "ë": "e",
        "í": "i", "ì": "i", "î": "i", "ï": "i",
        "ó": "o", "ò": "o", "ô": "o", "ö": "o", "õ": "o",
        "ú": "u", "ù": "u", "û": "u", "ü": "u",
        "ñ": "n",
    }
    for a, b in repl.items():
        text = text.replace(a, b)
    text = re.sub(r"[^a-z0-9]+", "_", text)
    return text.strip("_")


def clean_value(value: Any) -> Any:
    if value is None:
        return ""
    if isinstance(value, datetime):
        return value.date().isoformat()
    if isinstance(value, date):
        return value.isoformat()
    if isinstance(value, str):
        return value.strip()
    return value


def read_table(ws) -> list[dict[str, Any]]:
    headers = [slug_header(ws.cell(1, c).value) for c in range(1, ws.max_column + 1)]
    rows = []
    for r in range(2, ws.max_row + 1):
        row = {}
        has_data = False
        for c, header in enumerate(headers, start=1):
            if not header:
                continue
            value = clean_value(ws.cell(r, c).value)
            if value not in ("", None):
                has_data = True
            row[header] = value
        if has_data:
            rows.append(row)
    return rows


def first(row: dict[str, Any], *keys: str) -> Any:
    for key in keys:
        if key in row and row[key] not in ("", None):
            return row[key]
    return ""


def build_search_text(item: dict[str, Any]) -> str:
    parts = []

    def walk(value: Any):
        if isinstance(value, dict):
            for v in value.values():
                walk(v)
        elif isinstance(value, list):
            for v in value:
                walk(v)
        elif value not in (None, ""):
            parts.append(str(value))

    walk(item)
    return " | ".join(parts)


def read_listas(wb) -> dict[str, list[str]]:
    if SHEET_LISTAS not in wb.sheetnames:
        return {}
    ws = wb[SHEET_LISTAS]
    listas = {}
    headers = [slug_header(ws.cell(1, c).value) for c in range(1, ws.max_column + 1)]
    for idx, header in enumerate(headers, start=1):
        if not header:
            continue
        values = []
        for r in range(2, ws.max_row + 1):
            value = clean_value(ws.cell(r, idx).value)
            if value not in ("", None):
                values.append(str(value))
        listas[header] = values
    return listas


def main() -> None:
    parser = argparse.ArgumentParser(description="Gera JSON do Wolf Troubleshooting.")
    parser.add_argument("--excel", default=str(DEFAULT_EXCEL), help="Caminho da planilha .xlsx")
    parser.add_argument("--saida", default=str(DEFAULT_SAIDA), help="Pasta de saída")
    args, _unknown = parser.parse_known_args()

    excel_path = Path(args.excel)
    out_dir = Path(args.saida)
    out_dir.mkdir(parents=True, exist_ok=True)

    if not excel_path.exists():
        raise FileNotFoundError(f"Planilha não encontrada: {excel_path}")

    wb = openpyxl.load_workbook(excel_path, data_only=True)

    required = [SHEET_TROUBLE, SHEET_DIAG, SHEET_ANEXOS, SHEET_REVISOES]
    missing = [s for s in required if s not in wb.sheetnames]
    if missing:
        raise RuntimeError(f"Abas obrigatórias não encontradas: {missing}")

    trouble_rows = read_table(wb[SHEET_TROUBLE])
    diag_rows = read_table(wb[SHEET_DIAG])
    anexos_rows = read_table(wb[SHEET_ANEXOS])
    revisoes_rows = read_table(wb[SHEET_REVISOES])
    listas = read_listas(wb)

    avisos = []
    itens_by_codigo = {}

    for row in trouble_rows:
        codigo = str(first(row, "codigo", "id_falha_alerta", "id_item")).strip()
        if not codigo:
            continue
        if codigo in itens_by_codigo:
            avisos.append(f"Código duplicado na aba 01_Troubleshooting: {codigo}")
            continue

        item = {
            "id_item": codigo,
            "codigo": codigo,
            "identificacao": {
                "codigo": codigo,
                "tipo": first(row, "tipo"),
                "sistema": first(row, "sistema"),
                "subsistema": first(row, "subsistema"),
                "origem": first(row, "origem"),
                "variavel_codigo_relacionado": first(row, "variavel_codigo_relacionado", "variavel_codigo_relacionado_", "variavel_codigo_relacionado"),
            },
            "classificacao": {
                "severidade": first(row, "severidade"),
                "prioridade": first(row, "prioridade"),
                "status": first(row, "status"),
            },
            "conteudo": {
                "mensagem_ihm_alerta": first(row, "mensagem_na_ihm_alerta", "mensagem_ihm_alerta"),
                "descricao_resumida": first(row, "descricao_resumida"),
                "causa": first(row, "causa"),
                "efeito_maquina_operacao": first(row, "efeito_na_maquina_operacao"),
                "solucao": first(row, "solucao"),
                "validacao_apos_solucao": first(row, "validacao_apos_solucao"),
                "ferramentas_necessarias": first(row, "ferramentas_necessarias"),
                "observacoes": first(row, "observacoes"),
            },
            "controle": {
                "responsavel": first(row, "responsavel"),
                "data_revisao": first(row, "data_revisao"),
                "versao": first(row, "versao"),
            },
            "diagnostico_guiado": [],
            "anexos": [],
            "revisoes": [],
            "search_text": "",
        }
        itens_by_codigo[codigo] = item

    for row in diag_rows:
        codigo = str(first(row, "codigo_item", "id_falha_alerta", "codigo", "id_item")).strip()
        if not codigo:
            continue
        if codigo not in itens_by_codigo:
            avisos.append(f"Diagnóstico órfão: {first(row, 'id_passo')} referencia código inexistente {codigo}")
            continue

        # Suporta modelo novo Cummins e modelo antigo.
        passo = {
            "codigo_item": codigo,
            "id_passo": first(row, "id_passo"),
            "ordem": first(row, "ordem") or 0,
            "nivel": first(row, "nivel"),
            "titulo_passo": first(row, "titulo_do_passo", "pergunta_teste"),
            "condicoes": first(row, "condicoes"),
            "acao": first(row, "acao", "pergunta_teste"),
            "especificacao_pergunta": first(row, "especificacao_pergunta", "resposta_esperada"),
            "se_sim_ok": first(row, "se_sim_ok", "se_ok"),
            "se_nao_nok": first(row, "se_nao_nok", "se_nao_ok"),
            "proximo_passo_sim": first(row, "proximo_passo_sim"),
            "proximo_passo_nao": first(row, "proximo_passo_nao"),
            "reparo_correcao": first(row, "reparo_correcao"),
            "ferramentas": first(row, "ferramentas"),
            "seguranca": first(row, "seguranca"),
            "observacoes": first(row, "observacoes"),
        }
        itens_by_codigo[codigo]["diagnostico_guiado"].append(passo)

    for row in anexos_rows:
        codigo = str(first(row, "codigo_item", "id_falha_alerta", "codigo", "id_item")).strip()
        if not codigo:
            continue
        if codigo not in itens_by_codigo:
            avisos.append(f"Anexo órfão: {first(row, 'id_anexo')} referencia código inexistente {codigo}")
            continue
        anexo = {
            "id_anexo": first(row, "id_anexo"),
            "codigo_item": codigo,
            "tipo": first(row, "tipo"),
            "titulo": first(row, "titulo"),
            "arquivo_caminho_relativo": first(row, "arquivo_caminho_relativo"),
            "descricao": first(row, "descricao"),
            "ordem": first(row, "ordem") or 0,
            "observacoes": first(row, "observacoes"),
        }
        itens_by_codigo[codigo]["anexos"].append(anexo)

    for row in revisoes_rows:
        codigos = str(first(row, "itens_alterados", "codigo_item", "codigo")).strip()
        if not codigos:
            continue
        for codigo in re.split(r"[,;|]+", codigos):
            codigo = codigo.strip()
            if not codigo:
                continue
            if codigo not in itens_by_codigo:
                avisos.append(f"Revisão órfã: referencia código inexistente {codigo}")
                continue
            revisao = {
                "versao": first(row, "versao"),
                "data": first(row, "data"),
                "responsavel": first(row, "responsavel"),
                "tipo_alteracao": first(row, "tipo_alteracao"),
                "descricao": first(row, "descricao"),
                "itens_alterados": codigos,
                "status_publicacao": first(row, "status_publicacao"),
            }
            itens_by_codigo[codigo]["revisoes"].append(revisao)

    itens = list(itens_by_codigo.values())
    for item in itens:
        item["diagnostico_guiado"].sort(key=lambda x: (int(x.get("ordem") or 0), str(x.get("id_passo") or "")))
        item["anexos"].sort(key=lambda x: int(x.get("ordem") or 0))
        item["search_text"] = build_search_text(item)

    metadata_source = itens[0]["controle"] if itens else {}
    json_data = {
        "metadata": {
            "produto": "Wolf Troubleshooting de Alertas e Falhas",
            "formato": "dados_troubleshooting",
            "schema_version": "2.0.0",
            "modelo_diagnostico": "cummins_like_decision_flow",
            "versao_base": metadata_source.get("versao") or "0.1.0",
            "data_referencia_base": metadata_source.get("data_revisao") or date.today().isoformat(),
            "data_geracao_json": datetime.now().replace(microsecond=0).isoformat(),
            "arquivo_fonte": str(excel_path),
            "chave_correlacao": "codigo",
            "observacao": "Arquivo JSON gerado a partir da planilha pai. O diagnóstico guiado usa um modelo de árvore de decisão inspirado no padrão Cummins.",
            "totais": {
                "itens_troubleshooting": len(itens),
                "passos_diagnostico": sum(len(i["diagnostico_guiado"]) for i in itens),
                "anexos": sum(len(i["anexos"]) for i in itens),
                "revisoes": sum(len(i["revisoes"]) for i in itens),
            },
            "avisos_validacao": avisos,
        },
        "configuracao_interface": {
            "campos_pesquisa": [
                "codigo", "mensagem_ihm_alerta", "descricao_resumida", "causa", "efeito_maquina_operacao", "solucao", "sistema", "subsistema", "origem", "variavel_codigo_relacionado", "search_text"
            ],
            "filtros_principais": ["tipo", "sistema", "severidade", "prioridade", "status", "origem"],
            "ordenacao_padrao": ["sistema", "severidade", "codigo"],
            "status_publicaveis_sugeridos": ["Homologado", "Publicado"],
            "modo_operacao": "offline",
        },
        "listas": listas,
        "itens": itens,
    }

    content_without_hash = json.dumps(json_data, ensure_ascii=False, indent=2, sort_keys=False)
    sha = hashlib.sha256(content_without_hash.encode("utf-8")).hexdigest()
    json_data["metadata"]["sha256_conteudo"] = sha

    dados_path = out_dir / "dados_troubleshooting.json"
    dados_path.write_text(json.dumps(json_data, ensure_ascii=False, indent=2), encoding="utf-8")

    schema = {
        "$schema": "https://json-schema.org/draft/2020-12/schema",
        "title": "Schema - Wolf Troubleshooting de Alertas e Falhas",
        "type": "object",
        "required": ["metadata", "configuracao_interface", "listas", "itens"],
        "properties": {
            "metadata": {"type": "object"},
            "configuracao_interface": {"type": "object"},
            "listas": {"type": "object"},
            "itens": {"type": "array"},
        },
    }
    (out_dir / "schema_troubleshooting.json").write_text(json.dumps(schema, ensure_ascii=False, indent=2), encoding="utf-8")

    relatorio = [
        "RELATORIO DE GERACAO DO JSON - WOLF TROUBLESHOOTING",
        "=" * 62,
        f"Data/hora geracao: {json_data['metadata']['data_geracao_json']}",
        f"Arquivo fonte: {excel_path}",
        f"Modelo diagnostico: {json_data['metadata']['modelo_diagnostico']}",
        f"Chave de correlacao entre abas: codigo",
        "",
        "Totais:",
        f"- Itens troubleshooting: {json_data['metadata']['totais']['itens_troubleshooting']}",
        f"- Passos diagnostico: {json_data['metadata']['totais']['passos_diagnostico']}",
        f"- Anexos: {json_data['metadata']['totais']['anexos']}",
        f"- Revisoes: {json_data['metadata']['totais']['revisoes']}",
        "",
        f"SHA-256 conteudo: {sha}",
        "",
        "Avisos de validacao:",
    ]
    relatorio.extend([f"- {a}" for a in avisos] or ["- Nenhum aviso encontrado."])
    (out_dir / "relatorio_geracao_json.txt").write_text("\n".join(relatorio), encoding="utf-8")

    print(f"JSON gerado em: {dados_path}")
    print(f"Avisos: {len(avisos)}")


if __name__ == "__main__":
    main()


JSON gerado em: C:\Users\wesley\Documents\13 - TroubleShooting\99 - Saidas JSON\dados_troubleshooting.json
Avisos: 12


C:\Users\wesley\AppData\Local\anaconda3\Lib\site-packages\openpyxl\reader\excel.py:237: UserWarning: Data Validation extension is not supported and will be removed
  ws_parser.bind_all()
